# 🚀 InterviewIQ-Coder: Fine-Tuning Qwen 2.5 Coder on Free Google Colab GPU

Bu notebook **InterviewIQ AI** üçün açıq mənbəli **Qwen 2.5 Coder** modelini **Unsloth (QLoRA)** ilə sürətli və yaddaşa qənaət edən şəkildə train (fine-tune) edir və sonda modeli birbaşa **GGUF** formatında ixrac edir ki, yerli **Ollama** mühitində (`RTX 3050 GPU`) işə salasınız.

---

### 1. Unsloth və Asılılıqların Quraşdırılması

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl" "peft" "accelerate" "bitsandbytes"

### 2. Əsas Modelin Yüklənməsi (Qwen 2.5 Coder 3B və ya 7B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 2048 token kontekst pəncərəsi
dtype = None # Avtomatik float16/bfloat16 seçimi
load_in_4bit = True # 4-bit QLoRA kvantlaşdırması (VRAM-a qənaət)

# 3B model RTX 3050-də inanılmaz sürətlə işləyir; istəsəniz 7B də seçə bilərsiniz:
model_name = "unsloth/Qwen2.5-Coder-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. LoRA / PEFT Adapterlərinin Əlavə Edilməsi

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA dərəcəsi
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 30% daha az VRAM istifadəsi
    random_state = 3407,
)

### 4. Datasetin Yüklənməsi (`interviewiq_coder_sft.jsonl`)
Sol paneldən faylı Colab-a yükləyin və ya birbaşa oxudun.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen və Llama üçün standart ChatML şablonu
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Faylın yolu
dataset = load_dataset("json", data_files = "interviewiq_coder_sft.jsonl", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

### 5. Modelin Təlimi (Training with SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Təlim addımı (dataset həcminə görə artırıla bilər)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 6. Modelin GGUF Formatında İxracı (Ollama üçün)

In [ ]:
# Modeli q4_k_m kvantlaşdırma ilə GGUF formatına çeviririk (Ollama və RTX 3050 üçün ideal)
model.save_pretrained_gguf("interviewiq_coder_model", tokenizer, quantization_method = "q4_k_m")

print("✅ Model uğurla GGUF formatına çevrildi! Sol paneldən faylı kompüterinizə endirin.")